In [35]:
from src.connection.clinets import *
from typing import List
import weaviate
from langchain_weaviate import WeaviateVectorStore
from sentence_transformers import SentenceTransformer

In [38]:
class SentenceTransformersEmbeddings:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # returns a list of embeddings for documents
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # returns a single embedding for a query
        return self.model.encode([text])[0].tolist()

In [39]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 583.06it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.encode([enhanced_query]).tolist()[0]

    return query_embedding

In [26]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=WEAVIATE_API_KEY,
)

In [27]:
Euro_Laws = weaviate_client.collections.use("Euro_Laws")

# Step 2.3: Perform a vector search with NearVector
response = Euro_Laws.query.near_vector(
    near_vector= query_embedding, 
    limit=5
)


True

In [41]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [42]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

docs = retriever.invoke("Drug dealing Sentences")

for doc in docs:
    print(">>", doc.page_content)

>> 11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Article 31(e) and Article 34(2)(b) thereof, Having regard to the proposal from the Commission (1), Having regard to the opinion of the European Parliament (2), Whereas: (1) Illicit drug trafficking poses a threat to health, safety and the quality of life of citizens of the European Union, and to the legal economy, stability and security of the Member States. (2) The need for legislative action to tackle illicit drug trafficking has been recognised in particular in the Action Plan of the Council and the Commission on how best to implement the provisions of the Amsterdam Treaty on an area of freedom, security and justice (3), adopted by the

In [45]:
docs[0].metadata['celex']

'32004F0757'

In [ ]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(docs[i].metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids    